In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter


file_path = "Reviews.csv"


df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "snap/amazon-fine-food-reviews",
  file_path,
)

print("First 5 records:\n", df.head())

<h1>Importing NLTK & required models</h1>
<ul>
    <li>NLTK: Library for NLP</li>
    <li>stopwords: Helps removing words like "is", "a", "the" etc.</li>
    <li>vader_lexicon: Valence Aware Dictionary and Sentiment Reasoner. Model for sentinemnt analysis</li>
</ul>

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('vader_lexicon')

<h1>Task 1: Load a 10k Sample</h1>

In [ ]:
df_sample = df.sample(n=10000, random_state=42).copy()
df_sample = df_sample.dropna(subset=['Text', 'Score'])
print(f"Sample shape: {df_sample.shape}")

<h1>Task 2: Clean text: lowercase, remove HTML tags, punctuation, stopwords using NLTK</h1>

In [ ]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    
    text = str(text).lower()
    
    text = re.sub(r'<.*?>', ' ', text)
    
    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    cleaned_words = [w for w in words if w not in stop_words]
    
    return ' '.join(cleaned_words)


df_sample['Cleaned_Text'] = df_sample['Text'].apply(clean_text)

print(df_sample[['Text', 'Cleaned_Text']].head(5))

<h1>Task 3: Apply VADER SentimentIntensityAnalyzer to get compound scores</h1>
<h3>Compound score ranges from +1 to -1.</h3>
<ul>
    <li>+1 represensts fulll positive</li>
    <li>-1 represensts fulll negative</li>
    <li>0 represensts neutral</li>
</ul>

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer


Senti = SentimentIntensityAnalyzer()


df_sample['Compound_Score'] = df_sample['Cleaned_Text'].apply(lambda x: Senti.polarity_scores(x)['compound'])

print(df_sample[['Cleaned_Text', 'Compound_Score']].head())

<h1>Task 4: Classify into Positive/Negative/Neutral based on compound score threshold</h1>

In [ ]:
def categorize_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'


df_sample['Vader_Sentiment'] = df_sample['Compound_Score'].apply(categorize_sentiment)

print(df_sample['Vader_Sentiment'].value_counts())

<h1>Task 5:Compare sentiment label with Star Rating — find mismatches</h1>

In [ ]:
def map_stars_to_sentiment(stars):
    if stars >= 4:
        return 'Positive'
    elif stars <= 2:
        return 'Negative'
    else:
        return 'Neutral' 

df_sample['True_Sentiment'] = df_sample['Score'].apply(map_stars_to_sentiment)


mismatches = df_sample[df_sample['Vader_Sentiment'] != df_sample['True_Sentiment']]

print(f"Total Mismatches: {len(mismatches)}")
print("\nExample Mismatch:")
print(mismatches[['Score', 'Text', 'Vader_Sentiment']].iloc[0])

<h1>Task 6: Generate a word cloud for positive and negative reviews separately</h1>

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

positive_text = " ".join(review for review in df_sample[df_sample['Vader_Sentiment'] == 'Positive']['Cleaned_Text'])
negative_text = " ".join(review for review in df_sample[df_sample['Vader_Sentiment'] == 'Negative']['Cleaned_Text'])


wc_pos = WordCloud(width=400, height=300, background_color='white', colormap='Greens').generate(positive_text)
wc_neg = WordCloud(width=400, height=300, background_color='black', colormap='Reds').generate(negative_text)


fig, ax = plt.subplots(1, 2, figsize=(15, 6))

ax[0].imshow(wc_pos, interpolation='bilinear')
ax[0].set_title('Positive Words')
ax[0].axis('off')

ax[1].imshow(wc_neg, interpolation='bilinear')
ax[1].set_title('Negative Words')
ax[1].axis('off')

plt.show()

<h1>Task 7: Plot distribution of sentiment scores by star rating (boxplot)</h1>

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.boxplot(x='Score', y='Compound_Score', data=df_sample, palette='Set3')

plt.title('Distribution of VADER Compound Scores by Star Rating')
plt.xlabel('Amazon Star Rating (1-5)')
plt.ylabel('VADER Compound Score (-1 to 1)')
plt.show()

<h1>Task 8: Find top 20 most common positive and negative words using Counter</h1>

In [ ]:
from collections import Counter

pos_words = positive_text.split()
neg_words = negative_text.split()

pos_counter = Counter(pos_words)
neg_counter = Counter(neg_words)

print("Top 20 Positive Words:")
for word, count in pos_counter.most_common(20):
    print(f"{word}: {count}")

print("\n-----------------------\n")

print("Top 20 Negative Words:")
for word, count in neg_counter.most_common(20):
    print(f"{word}: {count}")